# Avaliação de Desempenho: Poda Estruturada de Camadas (*Layer Pruning*) no AnyGraph

Este notebook avalia o impacto da **Poda Estruturada de Camadas (*Layer Pruning*)** baseada no **Block Influence (BI)** nos modelos pré-treinados do AnyGraph (`pretrain_link1` e `pretrain_link2`), comparando o desempenho do modelo original com as versões reduzidas.

### Configurações Avaliadas:
1. **Modelo Inteiro (Baseline):** 8 camadas ativas por expert (0% de poda — 16.87M parâmetros).
2. **Poda Tier 1 (Conservadora):** Remoção das Camadas **`[5, 6]`** em todos os experts (**-25% de parâmetros** — 12.65M parâmetros).
3. **Poda Tier 2 (Agressiva):** Remoção das Camadas **`[4, 5, 6, 7]`** em todos os experts (**-50% de parâmetros** — 8.43M parâmetros).

### Datasets Oficiais Utilizados:
Utiliza os **5 datasets oficiais de classificação de nós do AnyGraph** (os mesmos do benchmark de *Global Magnitude Pruning*):
- **`arxiv`** (169.383 nós — artigos científicos)
- **`cora`** (25.191 nós — citações acadêmicas)
- **`home`** (9.795 nós — co-compra de produtos domésticos)
- **`pubmed`** (19.720 nós — publicações biomédicas)
- **`tech`** (47.431 nós — co-compra de tecnologia)

In [1]:
import os
import sys
import copy
import torch as t
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Identifica o diretorio OpenGraph de forma portavel
cwd = os.getcwd()
if os.path.basename(cwd) == "OpenGraph":
    PROJECT_ROOT = cwd
elif os.path.exists(os.path.join(cwd, "OpenGraph")):
    PROJECT_ROOT = os.path.join(cwd, "OpenGraph")
else:
    PROJECT_ROOT = os.path.abspath(".")

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.argv = ["pruning_evaluation"]

from node_classification.params import args

# Usa GPU para velocidade máxima de avaliação
if t.cuda.is_available():
    args.gpu = "0"
    args.devices = ["cuda:0", "cuda:0"]
    print(f"Utilizando Dispositivo: {args.devices[0]} (GPU)")
else:
    args.gpu = "-1"
    args.devices = ["cpu", "cpu"]
    print("Utilizando Dispositivo: CPU")

args.tst_mode = "tst"
args.trn_mode = "train-all"
args.shot = 0
args.proj_method = "both"
args.gnn_layer = 3
args.latdim = 512

import node_classification.Utils.TimeLogger as logger
from node_classification.Utils.TimeLogger import log

from node_classification.data_handler import *
import node_classification.model
from node_classification.main import *

from Pruning_Methods.layer_pruner import *
sys.modules["model"] = node_classification.model
# Diretorios de saida
OUT_DIR = os.path.join(os.path.abspath(".."), "Block_Influence_results", "layer_pruning_evaluation")
os.makedirs(OUT_DIR, exist_ok=True)

# Datasets oficiais do benchmark
OFFICIAL_NODE_DATASETS = ["cora", "pubmed", "citeseer"]
print(f"Datasets Oficiais de Avaliação: {OFFICIAL_NODE_DATASETS}")
print(f"Diretório de Resultados: {OUT_DIR}")

Utilizando Dispositivo: cuda:0 (GPU)
Datasets Oficiais de Avaliação: ['cora', 'pubmed', 'citeseer']
Diretório de Resultados: c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation


## 1. Pipeline Automatizado de Poda e Avaliação de Desempenho

In [17]:
def run_layer_pruning_experiment(
    model_checkpoint="pretrn_gen0",
    datasets=OFFICIAL_NODE_DATASETS,
    repeat_times=10
):
    """
    Executa Layer Pruning estruturado no OpenGraph
    baseado em Block Influence.

    Compara:
      1. Modelo Inteiro - 4 GTLayers
      2. Tier 1 - remove 1 camada com menor BI (25%)
      3. Tier 2 - remove 2 camadas com menor BI (50%)

    O Block Influence é recalculado para cada dataset.
    """

    print("=" * 80)
    print(
        f"EXPERIMENTO DE LAYER PRUNING - "
        f"CHECKPOINT: {model_checkpoint}"
    )
    print(f"Datasets: {datasets}")
    print("=" * 80)

    args.load_model = model_checkpoint

    # -------------------------------------------------------------
    # Preparação
    # -------------------------------------------------------------

    handler_data = MultiDataHandler(
        datasets,
        datasets
    )

    exp = Exp(handler_data)
    exp.prepare_model()

    results = []

    # -------------------------------------------------------------
    # Percorre cada dataset
    # -------------------------------------------------------------

    for handler in exp.multi_handler.tst_handlers:

        dset_tag = handler.data_name.lower()

        print("\n" + "=" * 80)
        print(f"DATASET: {dset_tag}")
        print("=" * 80)

        # =========================================================
        # 1. BASELINE
        # =========================================================

        print(
            f"\n[1/3] Baseline - "
            f"Modelo completo (4 GTLayers)"
        )

        exp.load_model()

        baseline_results = []

        for i in range(repeat_times):

            result = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            baseline_results.append(result)

        baseline_acc = np.mean(
            [r["Acc"] for r in baseline_results]
        )

        baseline_acc_std = np.std(
            [r["Acc"] for r in baseline_results]
        )

        baseline_f1 = np.mean(
            [r["F1"] for r in baseline_results]
        )

        baseline_f1_std = np.std(
            [r["F1"] for r in baseline_results]
        )

        results.append({
            "Configuração": "Modelo Inteiro (Baseline)",
            "Camadas Removidas": "Nenhuma",
            "Camadas Restantes": 4,
            "Redução Parâmetros (%)": 0.0,
            "Dataset": dset_tag,
            "Acc_mean": baseline_acc,
            "Acc_std": baseline_acc_std,
            "F1_mean": baseline_f1,
            "F1_std": baseline_f1_std
        })

        print(
            f"Baseline | "
            f"Acc: {baseline_acc:.4f} ± {baseline_acc_std:.4f} | "
            f"F1: {baseline_f1:.4f} ± {baseline_f1_std:.4f}"
        )

        # =========================================================
        # Calcular BI
        # =========================================================

        print("\nCalculando Block Influence...")

        adj = handler.torch_adj
        initial_projector = handler.initial_projector

        if args.cache_adj == 0:
            adj = adj.to(args.devices[0])

        if args.cache_proj == 0:
            initial_projector = initial_projector.to(
                args.devices[0]
            )

        with t.no_grad():

            initial_embeds = initial_projector()

            input_embeds = exp.model.topoEncoder(
                adj,
                initial_embeds
            ).to(args.devices[1])

        pruner = BlockInfluenceLayerPruner(
            exp.model
        )

        df_bi = pruner.compute_bi(
            input_embeds,
            angular=False
        )

        print("\nBlock Influence:")
        display(df_bi)

        # Ranking das camadas
        ranking = (
            df_bi
            .sort_values("Block Influence")
            ["Layer"]
            .tolist()
        )

        print(
            "Ranking das camadas "
            "(menor → maior BI):",
            ranking
        )

        # =========================================================
        # 2. TIER 1
        # =========================================================

        layers_t1 = ranking[:1]

        print(
            f"\n[2/3] Tier 1 - "
            f"Removendo camada {layers_t1}"
        )

        # Recarrega modelo original
        exp.load_model()

        pruner_t1 = BlockInfluenceLayerPruner(
            exp.model
        )

        pruned_model_t1 = (
            pruner_t1.prune_specific_layers(
                layers_t1
            )
        )

        exp.model = pruned_model_t1

        # Verificação
        report_t1 = pruner_t1.sparsity_report()

        print(
            f"Parâmetros: "
            f"{report_t1['Total Params Original']:,} → "
            f"{report_t1['Total Params Pruned']:,}"
        )

        print(
            f"Redução real: "
            f"{report_t1['Total Sparsity (%)']:.2f}%"
        )

        t1_results = []

        for i in range(repeat_times):

            result = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            t1_results.append(result)

        t1_acc = np.mean(
            [r["Acc"] for r in t1_results]
        )

        t1_acc_std = np.std(
            [r["Acc"] for r in t1_results]
        )

        t1_f1 = np.mean(
            [r["F1"] for r in t1_results]
        )

        t1_f1_std = np.std(
            [r["F1"] for r in t1_results]
        )

        results.append({
            "Configuração": "Poda Tier 1",
            "Camadas Removidas": str(layers_t1),
            "Camadas Restantes": 3,
            "Redução Parâmetros (%)":
                report_t1["Total Sparsity (%)"],
            "Dataset": dset_tag,
            "Acc_mean": t1_acc,
            "Acc_std": t1_acc_std,
            "F1_mean": t1_f1,
            "F1_std": t1_f1_std
        })

        print(
            f"Tier 1 | "
            f"Acc: {t1_acc:.4f} ± {t1_acc_std:.4f} | "
            f"F1: {t1_f1:.4f} ± {t1_f1_std:.4f}"
        )

        # =========================================================
        # 3. TIER 2
        # =========================================================

        layers_t2 = ranking[:2]

        print(
            f"\n[3/3] Tier 2 - "
            f"Removendo camadas {layers_t2}"
        )

        # Recarrega modelo original
        exp.load_model()

        pruner_t2 = BlockInfluenceLayerPruner(
            exp.model
        )

        pruned_model_t2 = (
            pruner_t2.prune_specific_layers(
                layers_t2
            )
        )

        exp.model = pruned_model_t2

        # Verificação
        report_t2 = pruner_t2.sparsity_report()

        print(
            f"Parâmetros: "
            f"{report_t2['Total Params Original']:,} → "
            f"{report_t2['Total Params Pruned']:,}"
        )

        print(
            f"Redução real: "
            f"{report_t2['Total Sparsity (%)']:.2f}%"
        )

        t2_results = []

        for i in range(repeat_times):

            result = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            t2_results.append(result)

        t2_acc = np.mean(
            [r["Acc"] for r in t2_results]
        )

        t2_acc_std = np.std(
            [r["Acc"] for r in t2_results]
        )

        t2_f1 = np.mean(
            [r["F1"] for r in t2_results]
        )

        t2_f1_std = np.std(
            [r["F1"] for r in t2_results]
        )

        results.append({
            "Configuração": "Poda Tier 2",
            "Camadas Removidas": str(layers_t2),
            "Camadas Restantes": 2,
            "Redução Parâmetros (%)":
                report_t2["Total Sparsity (%)"],
            "Dataset": dset_tag,
            "Acc_mean": t2_acc,
            "Acc_std": t2_acc_std,
            "F1_mean": t2_f1,
            "F1_std": t2_f1_std
        })

        print(
            f"Tier 2 | "
            f"Acc: {t2_acc:.4f} ± {t2_acc_std:.4f} | "
            f"F1: {t2_f1:.4f} ± {t2_f1_std:.4f}"
        )

    # =============================================================
    # DataFrame final
    # =============================================================

    df_results = pd.DataFrame(results)

    print("\n" + "=" * 80)
    print("RESULTADOS FINAIS")
    print("=" * 80)

    display(df_results)

    # =============================================================
    # Salvar
    # =============================================================

    csv_path = os.path.join(
        OUT_DIR,
        f"desempenho_poda_camadas_{model_checkpoint}.csv"
    )

    df_results.to_csv(
        csv_path,
        index=False
    )

    print(
        f"\nResultados salvos em: {csv_path}"
    )

    return df_results

## 2. Avaliação de Poda de Camadas no Modelo 1 (`pretrn_gen0`)

In [18]:
df_res1 = run_layer_pruning_experiment("pretrn_gen0", OFFICIAL_NODE_DATASETS, repeat_times=10)
print("=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 1 ===")
display(df_res1)

EXPERIMENTO DE LAYER PRUNING - CHECKPOINT: pretrn_gen0
Datasets: ['cora', 'pubmed', 'citeseer']
Dataset: pubmed, Node num: 19720, Edge num: 89768
Dataset: citeseer, Node num: 3333, Edge num: 10344
Dataset: cora, Node num: 2715, Edge num: 11836
Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

DATASET: pubmed

[1/3] Baseline - Modelo completo (4 GTLayers)
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-10 17:25:15.311455: Model Loaded
Baseline | Acc: 0.0954 ± 0.0223 | F1: 0.0605 ± 0.0104= 1000           

Calculando Block Influence...

Block Influence:


,Layer,Block Influence
0,0,0.089064
1,1,0.023445
2,2,0.048424
3,3,0.073582


Ranking das camadas (menor → maior BI): [1, 2, 3, 0]

[2/3] Tier 1 - Removendo camada [1]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-10 17:25:17.389229: Model Loaded
Parâmetros: 25,190,400 → 18,892,800
Redução real: 25.00%
Tier 1 | Acc: 0.0794 ± 0.0221 | F1: 0.0487 ± 0.0097ot = 1000          

[3/3] Tier 2 - Removendo camadas [1, 2]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-10 17:25:19.282932: Model Loaded
Parâmetros: 25,190,400 → 12,595,200
Redução real: 50.00%
Tier 2 | Acc: 0.1183 ± 0.0606 | F1: 0.0627 ± 0.0191ot = 1000          

DATASET: citeseer

[1/3] Baseline - Modelo completo (4 GTLayers)
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\Histor

,Layer,Block Influence
0,0,0.065541
1,1,0.011214
2,2,0.026621
3,3,0.039337


Ranking das camadas (menor → maior BI): [1, 2, 3, 0]

[2/3] Tier 1 - Removendo camada [1]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-10 17:25:21.444807: Model Loaded
Parâmetros: 25,190,400 → 18,892,800
Redução real: 25.00%
Tier 1 | Acc: 0.0888 ± 0.0042 | F1: 0.0832 ± 0.0047t = 1000          

[3/3] Tier 2 - Removendo camadas [1, 2]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-10 17:25:22.364342: Model Loaded
Parâmetros: 25,190,400 → 12,595,200
Redução real: 50.00%
Tier 2 | Acc: 0.0906 ± 0.0082 | F1: 0.0836 ± 0.0076t = 1000           

DATASET: cora

[1/3] Baseline - Modelo completo (4 GTLayers)
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pre

,Layer,Block Influence
0,0,0.045333
1,1,0.014700
2,2,0.023051
3,3,0.039181


Ranking das camadas (menor → maior BI): [1, 2, 3, 0]

[2/3] Tier 1 - Removendo camada [1]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-10 17:25:23.703900: Model Loaded
Parâmetros: 25,190,400 → 18,892,800
Redução real: 25.00%
Tier 1 | Acc: 0.4913 ± 0.0159 | F1: 0.4757 ± 0.0136ot = 1000          

[3/3] Tier 2 - Removendo camadas [1, 2]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-10 17:25:24.417154: Model Loaded
Parâmetros: 25,190,400 → 12,595,200
Redução real: 50.00%
Tier 2 | Acc: 0.5223 ± 0.0266 | F1: 0.5109 ± 0.0236ot = 1000          

RESULTADOS FINAIS


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Modelo Inteiro (Baseline),Nenhuma,4,0.0,pubmed,0.0954,0.022339,0.060502,0.010411
1,Poda Tier 1,[1],3,25.0,pubmed,0.0794,0.022128,0.048696,0.009689
2,Poda Tier 2,"[1, 2]",2,50.0,pubmed,0.1183,0.060638,0.062685,0.019115
3,Modelo Inteiro (Baseline),Nenhuma,4,0.0,citeseer,0.0872,0.006431,0.083186,0.006332
4,Poda Tier 1,[1],3,25.0,citeseer,0.0888,0.004190,0.083235,0.004732
5,Poda Tier 2,"[1, 2]",2,50.0,citeseer,0.0906,0.008188,0.083569,0.007626
6,Modelo Inteiro (Baseline),Nenhuma,4,0.0,cora,0.4232,0.014959,0.418150,0.012882
7,Poda Tier 1,[1],3,25.0,cora,0.4913,0.015925,0.475669,0.013619
8,Poda Tier 2,"[1, 2]",2,50.0,cora,0.5223,0.026567,0.510941,0.023645



Resultados salvos em: c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation\desempenho_poda_camadas_pretrn_gen0.csv
=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 1 ===


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Modelo Inteiro (Baseline),Nenhuma,4,0.0,pubmed,0.0954,0.022339,0.060502,0.010411
1,Poda Tier 1,[1],3,25.0,pubmed,0.0794,0.022128,0.048696,0.009689
2,Poda Tier 2,"[1, 2]",2,50.0,pubmed,0.1183,0.060638,0.062685,0.019115
3,Modelo Inteiro (Baseline),Nenhuma,4,0.0,citeseer,0.0872,0.006431,0.083186,0.006332
4,Poda Tier 1,[1],3,25.0,citeseer,0.0888,0.004190,0.083235,0.004732
5,Poda Tier 2,"[1, 2]",2,50.0,citeseer,0.0906,0.008188,0.083569,0.007626
6,Modelo Inteiro (Baseline),Nenhuma,4,0.0,cora,0.4232,0.014959,0.418150,0.012882
7,Poda Tier 1,[1],3,25.0,cora,0.4913,0.015925,0.475669,0.013619
8,Poda Tier 2,"[1, 2]",2,50.0,cora,0.5223,0.026567,0.510941,0.023645


## 3. Avaliação de Poda de Camadas no Modelo 2 (`pretrn_gen1`)

In [19]:
df_res2 = run_layer_pruning_experiment("pretrn_gen1", OFFICIAL_NODE_DATASETS, repeat_times=10)
print("=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 2 ===")
display(df_res2)

EXPERIMENTO DE LAYER PRUNING - CHECKPOINT: pretrn_gen1
Datasets: ['cora', 'pubmed', 'citeseer']
Dataset: pubmed, Node num: 19720, Edge num: 89768
Dataset: citeseer, Node num: 3333, Edge num: 10344
Dataset: cora, Node num: 2715, Edge num: 11836


c:\Mestrado\Graph_Pruning\OpenGraph\node_classification\data_handler.py:62: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])


Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

DATASET: pubmed

[1/3] Baseline - Modelo completo (4 GTLayers)
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-10 17:27:46.281064: Model Loaded
Baseline | Acc: 0.0373 ± 0.0083 | F1: 0.0329 ± 0.0057= 1000          

Calculando Block Influence...

Block Influence:


,Layer,Block Influence
0,0,0.088131
1,1,0.027814
2,2,0.012529
3,3,0.006522


Ranking das camadas (menor → maior BI): [3, 2, 1, 0]

[2/3] Tier 1 - Removendo camada [3]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-10 17:27:48.258795: Model Loaded
Parâmetros: 25,190,400 → 18,892,800
Redução real: 25.00%
Tier 1 | Acc: 0.0450 ± 0.0144 | F1: 0.0369 ± 0.0080t = 1000          

[3/3] Tier 2 - Removendo camadas [3, 2]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-10 17:27:49.743789: Model Loaded
Parâmetros: 25,190,400 → 12,595,200
Redução real: 50.00%
Tier 2 | Acc: 0.0407 ± 0.0102 | F1: 0.0357 ± 0.0067t = 1000          

DATASET: citeseer

[1/3] Baseline - Modelo completo (4 GTLayers)
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\

...

,Layer,Block Influence
0,0,0.041362
1,1,0.016005
2,2,0.008385
3,3,0.005325


Ranking das camadas (menor → maior BI): [3, 2, 1, 0]

[2/3] Tier 1 - Removendo camada [3]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-10 17:27:52.501416: Model Loaded
Parâmetros: 25,190,400 → 18,892,800
Redução real: 25.00%
Tier 1 | Acc: 0.7487 ± 0.0045 | F1: 0.7468 ± 0.0041ot = 1000          

[3/3] Tier 2 - Removendo camadas [3, 2]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-10 17:27:52.854616: Model Loaded
Parâmetros: 25,190,400 → 12,595,200
Redução real: 50.00%
Tier 2 | Acc: 0.7464 ± 0.0045 | F1: 0.7444 ± 0.0049ot = 1000          

RESULTADOS FINAIS


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Modelo Inteiro (Baseline),Nenhuma,4,0.0,pubmed,0.0373,0.008319,0.032882,0.005750
1,Poda Tier 1,[3],3,25.0,pubmed,0.0450,0.014422,0.036899,0.007968
2,Poda Tier 2,"[3, 2]",2,50.0,pubmed,0.0407,0.010189,0.035734,0.006678
3,Modelo Inteiro (Baseline),Nenhuma,4,0.0,citeseer,0.0807,0.005832,0.075501,0.005849
4,Poda Tier 1,[3],3,25.0,citeseer,0.0870,0.006450,0.080703,0.006196
5,Poda Tier 2,"[3, 2]",2,50.0,citeseer,0.0841,0.006057,0.079197,0.004703
6,Modelo Inteiro (Baseline),Nenhuma,4,0.0,cora,0.7432,0.004020,0.739550,0.003957
7,Poda Tier 1,[3],3,25.0,cora,0.7487,0.004451,0.746836,0.004122
8,Poda Tier 2,"[3, 2]",2,50.0,cora,0.7464,0.004454,0.744404,0.004883



Resultados salvos em: c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation\desempenho_poda_camadas_pretrn_gen1.csv
=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 2 ===


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Modelo Inteiro (Baseline),Nenhuma,4,0.0,pubmed,0.0373,0.008319,0.032882,0.005750
1,Poda Tier 1,[3],3,25.0,pubmed,0.0450,0.014422,0.036899,0.007968
2,Poda Tier 2,"[3, 2]",2,50.0,pubmed,0.0407,0.010189,0.035734,0.006678
3,Modelo Inteiro (Baseline),Nenhuma,4,0.0,citeseer,0.0807,0.005832,0.075501,0.005849
4,Poda Tier 1,[3],3,25.0,citeseer,0.0870,0.006450,0.080703,0.006196
5,Poda Tier 2,"[3, 2]",2,50.0,citeseer,0.0841,0.006057,0.079197,0.004703
6,Modelo Inteiro (Baseline),Nenhuma,4,0.0,cora,0.7432,0.004020,0.739550,0.003957
7,Poda Tier 1,[3],3,25.0,cora,0.7487,0.004451,0.746836,0.004122
8,Poda Tier 2,"[3, 2]",2,50.0,cora,0.7464,0.004454,0.744404,0.004883


## 4. Avaliação de Poda de Camadas no Modelo 3 (`pretrn_gen2`)

In [20]:
df_res3 = run_layer_pruning_experiment("pretrn_gen2", OFFICIAL_NODE_DATASETS, repeat_times=10)
print("=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 3 ===")
display(df_res3)

EXPERIMENTO DE LAYER PRUNING - CHECKPOINT: pretrn_gen2
Datasets: ['cora', 'pubmed', 'citeseer']
Dataset: pubmed, Node num: 19720, Edge num: 89768
Dataset: citeseer, Node num: 3333, Edge num: 10344
Dataset: cora, Node num: 2715, Edge num: 11836


c:\Mestrado\Graph_Pruning\OpenGraph\node_classification\data_handler.py:62: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])


Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

DATASET: pubmed

[1/3] Baseline - Modelo completo (4 GTLayers)
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-10 17:29:49.448583: Model Loaded
Baseline | Acc: 0.0798 ± 0.0488 | F1: 0.0555 ± 0.0210 = 1000          

Calculando Block Influence...

Block Influence:


,Layer,Block Influence
0,0,0.113987
1,1,0.039852
2,2,0.020208
3,3,0.010143


Ranking das camadas (menor → maior BI): [3, 2, 1, 0]

[2/3] Tier 1 - Removendo camada [3]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-10 17:29:51.367161: Model Loaded
Parâmetros: 25,190,400 → 18,892,800
Redução real: 25.00%
Tier 1 | Acc: 0.0547 ± 0.0165 | F1: 0.0438 ± 0.0110t = 1000          

[3/3] Tier 2 - Removendo camadas [3, 2]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-10 17:29:52.801728: Model Loaded
Parâmetros: 25,190,400 → 12,595,200
Redução real: 50.00%
Tier 2 | Acc: 0.0591 ± 0.0528 | F1: 0.0425 ± 0.0232t = 1000           

DATASET: citeseer

[1/3] Baseline - Modelo completo (4 GTLayers)
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History

...

,Layer,Block Influence
0,0,0.053440
1,1,0.024827
2,2,0.014046
3,3,0.008111


Ranking das camadas (menor → maior BI): [3, 2, 1, 0]

[2/3] Tier 1 - Removendo camada [3]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-10 17:29:55.525425: Model Loaded
Parâmetros: 25,190,400 → 18,892,800
Redução real: 25.00%
Tier 1 | Acc: 0.7588 ± 0.0059 | F1: 0.7508 ± 0.0059ot = 1000          

[3/3] Tier 2 - Removendo camadas [3, 2]
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-10 17:29:55.870402: Model Loaded
Parâmetros: 25,190,400 → 12,595,200
Redução real: 50.00%
Tier 2 | Acc: 0.7607 ± 0.0097 | F1: 0.7531 ± 0.0080ot = 1000          

RESULTADOS FINAIS


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Modelo Inteiro (Baseline),Nenhuma,4,0.0,pubmed,0.0798,0.048803,0.055508,0.020988
1,Poda Tier 1,[3],3,25.0,pubmed,0.0547,0.016468,0.043756,0.011018
2,Poda Tier 2,"[3, 2]",2,50.0,pubmed,0.0591,0.052842,0.042487,0.023215
3,Modelo Inteiro (Baseline),Nenhuma,4,0.0,citeseer,0.0815,0.007159,0.075927,0.006828
4,Poda Tier 1,[3],3,25.0,citeseer,0.0787,0.005849,0.072811,0.005390
5,Poda Tier 2,"[3, 2]",2,50.0,citeseer,0.0787,0.006619,0.073282,0.005651
6,Modelo Inteiro (Baseline),Nenhuma,4,0.0,cora,0.7544,0.007486,0.747707,0.006465
7,Poda Tier 1,[3],3,25.0,cora,0.7588,0.005896,0.750815,0.005924
8,Poda Tier 2,"[3, 2]",2,50.0,cora,0.7607,0.009655,0.753059,0.007950



Resultados salvos em: c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation\desempenho_poda_camadas_pretrn_gen2.csv
=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 3 ===


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Modelo Inteiro (Baseline),Nenhuma,4,0.0,pubmed,0.0798,0.048803,0.055508,0.020988
1,Poda Tier 1,[3],3,25.0,pubmed,0.0547,0.016468,0.043756,0.011018
2,Poda Tier 2,"[3, 2]",2,50.0,pubmed,0.0591,0.052842,0.042487,0.023215
3,Modelo Inteiro (Baseline),Nenhuma,4,0.0,citeseer,0.0815,0.007159,0.075927,0.006828
4,Poda Tier 1,[3],3,25.0,citeseer,0.0787,0.005849,0.072811,0.005390
5,Poda Tier 2,"[3, 2]",2,50.0,citeseer,0.0787,0.006619,0.073282,0.005651
6,Modelo Inteiro (Baseline),Nenhuma,4,0.0,cora,0.7544,0.007486,0.747707,0.006465
7,Poda Tier 1,[3],3,25.0,cora,0.7588,0.005896,0.750815,0.005924
8,Poda Tier 2,"[3, 2]",2,50.0,cora,0.7607,0.009655,0.753059,0.007950
